In [41]:
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import *

In [42]:
spark = (SparkSession
        .builder
        .appName("Spark Higher Order Functions")
        .getOrCreate())

First we will setup some data to demonstrate higher order functions.

In [43]:
from pyspark.sql.types import *
schema = StructType([StructField("celsius", ArrayType(IntegerType()))])

In [44]:
t_list = [[35, 36, 32, 30, 40, 42, 38]], [[31, 32, 34, 55, 56]]
t_c = spark.createDataFrame(t_list, schema)
t_c.createOrReplaceTempView("tC")

In [45]:
# Show the DataFrame
t_c.show()

+--------------------+
|             celsius|
+--------------------+
|[35, 36, 32, 30, ...|
|[31, 32, 34, 55, 56]|
+--------------------+



## transform()
`transform(array<T>, function<T, U>): array<U>`

The transform() function produces an array by applying a function to each element
of the input array (similar to a map() function):


In [46]:
#Calculate Fahrenheit from Celsius for an array of temperatures

spark.sql("""
SELECT celsius,
    transform(celsius, t -> ((t * 9) div 5) + 32) as fahrenheit
    FROM tC
""").show()


+--------------------+--------------------+
|             celsius|          fahrenheit|
+--------------------+--------------------+
|[35, 36, 32, 30, ...|[95, 96, 89, 86, ...|
|[31, 32, 34, 55, 56]|[87, 89, 93, 131,...|
+--------------------+--------------------+



## filter()
`filter(array<T>, function<T, Boolean>): array<T>`

The `filter()` function produces an array consisting of only the elements of the input
array for which the Boolean function is true:


In [47]:
spark.sql("""
    SELECT celsius,
    filter(celsius, t -> t > 38) as high
    FROM tC
""").show()


+--------------------+--------+
|             celsius|    high|
+--------------------+--------+
|[35, 36, 32, 30, ...|[40, 42]|
|[31, 32, 34, 55, 56]|[55, 56]|
+--------------------+--------+



## exists()
`exists(array<T>, function<T, V, Boolean>): Boolean`

The `exists()` function returns true if the Boolean function holds for any element in
the input array:



In [48]:
spark.sql("""
    SELECT celsius,
    exists(celsius, t -> t = 38) as threshold
    FROM tC
""").show()

+--------------------+---------+
|             celsius|threshold|
+--------------------+---------+
|[35, 36, 32, 30, ...|     true|
|[31, 32, 34, 55, 56]|    false|
+--------------------+---------+



## reduce()
`reduce(array<T>, B, function<B, T, B>, function<B, R>)`

The `reduce()` function reduces the elements of the array to a single value by merging
the elements into a buffer B using `function<B, T, B>` and applying a finishing
`function<B, R>` on the final buffer:


In [49]:
spark.sql("""
    SELECT celsius,
    reduce(
    celsius,
    0,
    (t, acc) -> t + acc,
    acc -> (acc div size(celsius) * 9 div 5) + 32
    ) as avgFahrenheit
    FROM tC
""").show()

+--------------------+-------------+
|             celsius|avgFahrenheit|
+--------------------+-------------+
|[35, 36, 32, 30, ...|           96|
|[31, 32, 34, 55, 56]|          105|
+--------------------+-------------+



Let's set up some data for more higher order functions.

In [50]:
from pyspark.sql.functions import expr
tripdelaysFilePath = "../spark-sql-python/flights/data/departuredelays.csv"
airportsnaFilePath = "../spark-sql-python/flights/data/airport-codes-na.txt"

In [51]:
# Obtain airport data set
airportsna = (spark.read
.format("csv")
.options(header="true", inferSchema="true", sep="\t")
.load(airportsnaFilePath))

In [52]:
airportsna.createOrReplaceTempView("airports_na")

In [53]:
# Obtain departure delays data set
departureDelays = (spark.read
.format("csv")
.options(header="true")
.load(tripdelaysFilePath))

In [54]:
departureDelays = (departureDelays
.withColumn("delay", expr("CAST(delay as INT) as delay"))
.withColumn("distance", expr("CAST(distance as INT) as distance")))

departureDelays.createOrReplaceTempView("departureDelays")

In [55]:
table1 = (departureDelays
.filter(expr("""origin == 'SEA' and destination == 'SFO' and
date like '01010%' and delay > 0""")))

table1.createOrReplaceTempView("table1")

In [56]:
spark.sql("SELECT * FROM airports_na LIMIT 10").show()

+-----------+-----+-------+----+
|       City|State|Country|IATA|
+-----------+-----+-------+----+
| Abbotsford|   BC| Canada| YXX|
|   Aberdeen|   SD|    USA| ABR|
|    Abilene|   TX|    USA| ABI|
|      Akron|   OH|    USA| CAK|
|    Alamosa|   CO|    USA| ALS|
|     Albany|   GA|    USA| ABY|
|     Albany|   NY|    USA| ALB|
|Albuquerque|   NM|    USA| ABQ|
| Alexandria|   LA|    USA| AEX|
|  Allentown|   PA|    USA| ABE|
+-----------+-----+-------+----+



In [57]:
spark.sql("SELECT * FROM departureDelays LIMIT 10").show()

+--------+-----+--------+------+-----------+
|    date|delay|distance|origin|destination|
+--------+-----+--------+------+-----------+
|01011245|    6|     602|   ABE|        ATL|
|01020600|   -8|     369|   ABE|        DTW|
|01021245|   -2|     602|   ABE|        ATL|
|01020605|   -4|     602|   ABE|        ATL|
|01031245|   -4|     602|   ABE|        ATL|
|01030605|    0|     602|   ABE|        ATL|
|01041243|   10|     602|   ABE|        ATL|
|01040605|   28|     602|   ABE|        ATL|
|01051245|   88|     602|   ABE|        ATL|
|01050605|    9|     602|   ABE|        ATL|
+--------+-----+--------+------+-----------+



In [58]:
spark.sql("SELECT * FROM table1").show()

+--------+-----+--------+------+-----------+
|    date|delay|distance|origin|destination|
+--------+-----+--------+------+-----------+
|01010710|   31|     590|   SEA|        SFO|
|01010955|  104|     590|   SEA|        SFO|
|01010730|    5|     590|   SEA|        SFO|
+--------+-----+--------+------+-----------+



## Union

A common pattern within Apache Spark is to union two different DataFrames with
the same schema together. This can be achieved using the union() method:


In [59]:
# Union two tables
table2 = departureDelays.union(table1)
table2.createOrReplaceTempView("table2")
# Show the union (filtering for SEA and SFO in a specific time range)
table2.filter(expr("""origin == 'SEA' AND destination == 'SFO'
AND date LIKE '01010%' AND delay > 0""")).show()


+--------+-----+--------+------+-----------+
|    date|delay|distance|origin|destination|
+--------+-----+--------+------+-----------+
|01010710|   31|     590|   SEA|        SFO|
|01010955|  104|     590|   SEA|        SFO|
|01010730|    5|     590|   SEA|        SFO|
|01010710|   31|     590|   SEA|        SFO|
|01010955|  104|     590|   SEA|        SFO|
|01010730|    5|     590|   SEA|        SFO|
+--------+-----+--------+------+-----------+



In [60]:
spark.sql("""
SELECT *
FROM table2
WHERE origin = 'SEA'
AND destination = 'SFO'
AND date LIKE '01010%'
AND delay > 0
""").show()


+--------+-----+--------+------+-----------+
|    date|delay|distance|origin|destination|
+--------+-----+--------+------+-----------+
|01010710|   31|     590|   SEA|        SFO|
|01010955|  104|     590|   SEA|        SFO|
|01010730|    5|     590|   SEA|        SFO|
|01010710|   31|     590|   SEA|        SFO|
|01010955|  104|     590|   SEA|        SFO|
|01010730|    5|     590|   SEA|        SFO|
+--------+-----+--------+------+-----------+



## Joins
A common DataFrame operation is to join two DataFrames (or tables) together. By
default, a Spark SQL join is an inner join, with the options being inner, cross,
outer, full, full_outer, left, left_outer, right, right_outer, left_semi, and
left_anti. 

The following code sample performs the default of an inner join between the `airportsna` and `table1` DataFrames:


In [61]:
# Join departure delays data (foo) with airport info
table1.join(
airportsna,
airportsna.IATA == table1.origin
).select("City", "State", "date", "delay", "distance", "destination").show()

+-------+-----+--------+-----+--------+-----------+
|   City|State|    date|delay|distance|destination|
+-------+-----+--------+-----+--------+-----------+
|Seattle|   WA|01010710|   31|     590|        SFO|
|Seattle|   WA|01010955|  104|     590|        SFO|
|Seattle|   WA|01010730|    5|     590|        SFO|
+-------+-----+--------+-----+--------+-----------+



## Windowing
A window function uses values from the rows in a window (a range of input rows) to
return a set of values, typically in the form of another row. With window functions, it is possible to operate on a group of rows while still returning a single value for every input row. In this section, we will show how to use the dense_rank() window function; there are many other functions, as noted in the table below:


| Function Type | SQL | DataFrame API |
| :--- | :--- | :--- |
| Ranking functions | rank() | rank() |
| | dense_rank() | denseRank() |
| | percent_rank() | percentRank() |
| | ntile() | ntile() |
| | row_number() | rowNumber() |
| Analytic functions | cume_dist() | cumeDist() |
| | first_value() | firstValue() |
| | last_value() | lastValue() |
| | lag() | lag() |
| | lead() | lead() |



Let’s start with a review of the TotalDelays (calculated by sum(Delay)) experienced
by flights originating from Seattle (SEA), San Francisco (SFO), and New York City
(JFK) and going to a specific set of destination locations, as noted in the following
query:


In [68]:
spark.sql("""DROP TABLE IF EXISTS departureDelaysWindow;""")


DataFrame[]

In [69]:
spark.sql("""
CREATE TABLE departureDelaysWindow AS
SELECT origin, destination, SUM(delay) AS TotalDelays
FROM departureDelays
WHERE origin IN ('SEA', 'SFO', 'JFK')
AND destination IN ('SEA', 'SFO', 'JFK', 'DEN', 'ORD', 'LAX', 'ATL')
GROUP BY origin, destination;
""")

++
||
++
++



In [70]:
spark.sql("""SELECT * FROM departureDelaysWindow""").show()

+------+-----------+-----------+
|origin|destination|TotalDelays|
+------+-----------+-----------+
|   JFK|        ORD|       5608|
|   JFK|        SFO|      35619|
|   JFK|        DEN|       4315|
|   JFK|        ATL|      12141|
|   JFK|        SEA|       7856|
|   JFK|        LAX|      35755|
|   SEA|        LAX|       9359|
|   SFO|        ORD|      27412|
|   SFO|        DEN|      18688|
|   SFO|        SEA|      17080|
|   SEA|        SFO|      22293|
|   SFO|        ATL|       5091|
|   SEA|        DEN|      13645|
|   SEA|        ATL|       4535|
|   SEA|        ORD|      10041|
|   SFO|        JFK|      24100|
|   SFO|        LAX|      40798|
|   SEA|        JFK|       4667|
+------+-----------+-----------+



What if for each of these origin airports we wanted to find the three destinations that experienced the most delays? We could achieve this by running three different queries for each origin and then unioning the results together, like this:
```
-- In SQL
SELECT origin, destination, SUM(TotalDelays) AS TotalDelays
FROM departureDelaysWindow
WHERE origin = '[ORIGIN]'
GROUP BY origin, destination
ORDER BY SUM(TotalDelays) DESC
LIMIT 3
```

But a better approach would be to use a window function like dense_rank() to perform the following calculation:



In [71]:
spark.sql("""
SELECT origin, destination, TotalDelays, rank
FROM (
SELECT origin, destination, TotalDelays, dense_rank()
OVER (PARTITION BY origin ORDER BY TotalDelays DESC) as rank
FROM departureDelaysWindow
) t
WHERE rank <= 3
""").show()

+------+-----------+-----------+----+
|origin|destination|TotalDelays|rank|
+------+-----------+-----------+----+
|   JFK|        LAX|      35755|   1|
|   JFK|        SFO|      35619|   2|
|   JFK|        ATL|      12141|   3|
|   SEA|        SFO|      22293|   1|
|   SEA|        DEN|      13645|   2|
|   SEA|        ORD|      10041|   3|
|   SFO|        LAX|      40798|   1|
|   SFO|        ORD|      27412|   2|
|   SFO|        JFK|      24100|   3|
+------+-----------+-----------+----+



By using the dense_rank() window function, we can quickly ascertain that the destinations with the worst delays for the three origin cities were:

 * Seattle (SEA): San Francisco (SFO), Denver (DEN), and Chicago (ORD)
 * San Francisco (SFO): Los Angeles (LAX), Chicago (ORD), and New York (JFK)
 * New York (JFK): Los Angeles (LAX), San Francisco (SFO), and Atlanta (ATL)
   
It’s important to note that each window grouping needs to fit in a single executor and
will get composed into a single partition during execution. Therefore, we need to
ensure that our queries are not unbounded (i.e., limit the size of your window).


## Modifications
Another common operation is to perform modifications to the DataFrame. While
DataFrames themselves are immutable, we can modify them through operations that
create new, different DataFrames, with different columns, for example. (Recall from
earlier chapters that the underlying RDDs are immutable—i.e., they cannot be
changed—to ensure there is data lineage for Spark operations.) 

Let’s start with our previous small DataFrame example:


In [73]:
table1.show()

+--------+-----+--------+------+-----------+
|    date|delay|distance|origin|destination|
+--------+-----+--------+------+-----------+
|01010710|   31|     590|   SEA|        SFO|
|01010955|  104|     590|   SEA|        SFO|
|01010730|    5|     590|   SEA|        SFO|
+--------+-----+--------+------+-----------+



### Adding Columns

In [74]:
from pyspark.sql.functions import expr
table3 = (table1.withColumn(
"status",
expr("CASE WHEN delay <= 10 THEN 'On-time' ELSE 'Delayed' END")
))


In [75]:
table3.show()

+--------+-----+--------+------+-----------+-------+
|    date|delay|distance|origin|destination| status|
+--------+-----+--------+------+-----------+-------+
|01010710|   31|     590|   SEA|        SFO|Delayed|
|01010955|  104|     590|   SEA|        SFO|Delayed|
|01010730|    5|     590|   SEA|        SFO|On-time|
+--------+-----+--------+------+-----------+-------+



### Dropping columns
To drop a column, use the drop() method. For example, let’s remove the delay column as we now have a status column, added in the previous section:


In [76]:
table4 = table3.drop("delay")
table4.show()


+--------+--------+------+-----------+-------+
|    date|distance|origin|destination| status|
+--------+--------+------+-----------+-------+
|01010710|     590|   SEA|        SFO|Delayed|
|01010955|     590|   SEA|        SFO|Delayed|
|01010730|     590|   SEA|        SFO|On-time|
+--------+--------+------+-----------+-------+



### Renaming Columns

In [77]:

table5 = table4.withColumnRenamed("status", "flight_status")
table5.show()


+--------+--------+------+-----------+-------------+
|    date|distance|origin|destination|flight_status|
+--------+--------+------+-----------+-------------+
|01010710|     590|   SEA|        SFO|      Delayed|
|01010955|     590|   SEA|        SFO|      Delayed|
|01010730|     590|   SEA|        SFO|      On-time|
+--------+--------+------+-----------+-------------+



## Pivoting
When working with your data, sometimes you will need to swap the columns for the
rows—i.e., pivot your data.


In [79]:
spark.sql("""
SELECT destination, CAST(SUBSTRING(date, 0, 2) AS int) AS month, delay
FROM departureDelays
WHERE origin = 'SEA'
""").show()

+-----------+-----+-----+
|destination|month|delay|
+-----------+-----+-----+
|        ORD|    1|   92|
|        JFK|    1|   -7|
|        DFW|    1|   -5|
|        MIA|    1|   -3|
|        DFW|    1|   -3|
|        DFW|    1|    1|
|        ORD|    1|  -10|
|        DFW|    1|   -6|
|        DFW|    1|   -2|
|        ORD|    1|   -3|
|        ORD|    1|    0|
|        DFW|    1|   23|
|        DFW|    1|   36|
|        ORD|    1|  298|
|        JFK|    1|    4|
|        DFW|    1|    0|
|        MIA|    1|    2|
|        DFW|    1|    0|
|        DFW|    1|    0|
|        ORD|    1|   83|
+-----------+-----+-----+
only showing top 20 rows


Pivoting allows you to place names in the month column (instead of 1 and 2 you can
show Jan and Feb, respectively) as well as perform aggregate calculations (in this case average and max) on the delays by destination and month:


In [81]:
spark.sql("""
    SELECT * FROM (
    SELECT destination, CAST(SUBSTRING(date, 0, 2) AS int) AS month, delay
    FROM departureDelays WHERE origin = 'SEA'
    )
    PIVOT (
    CAST(AVG(delay) AS DECIMAL(4, 2)) AS AvgDelay, MAX(delay) AS MaxDelay
    FOR month IN (1 JAN, 2 FEB)
    )
    ORDER BY destination
""").show()

+-----------+------------+------------+------------+------------+
|destination|JAN_AvgDelay|JAN_MaxDelay|FEB_AvgDelay|FEB_MaxDelay|
+-----------+------------+------------+------------+------------+
|        ABQ|       19.86|         316|       11.42|          69|
|        ANC|        4.44|         149|        7.90|         141|
|        ATL|       11.98|         397|        7.73|         145|
|        AUS|        3.48|          50|       -0.21|          18|
|        BOS|        7.84|         110|       14.58|         152|
|        BUR|       -2.03|          56|       -1.89|          78|
|        CLE|       16.00|          27|        NULL|        NULL|
|        CLT|        2.53|          41|       12.96|         228|
|        COS|        5.32|          82|       12.18|         203|
|        CVG|       -0.50|           4|        NULL|        NULL|
|        DCA|       -1.15|          50|        0.07|          34|
|        DEN|       13.13|         425|       12.95|         625|
|        D

In [82]:
spark.stop()